# LeetCode 1079 — Letter Tile Possibilities bằng Python

Link đề: https://leetcode.com/problems/letter-tile-possibilities/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Backtracking, Counting, Combinatorics

Skeleton LeetCode:

```python
class Solution:
    def numTilePossibilities(self, tiles: str) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**: số **dãy chữ cái khác nhau, không rỗng** có thể xếp từ các ô chữ (mỗi ô dùng tối đa một lần). Điểm cốt lõi: các ô trùng chữ cái **không phân biệt**, nên phải đếm **dãy phân biệt** chứ không phải hoán vị các ô. Quay lui trên **bảng đếm chữ cái** (`Counter`) tự tránh trùng; hoặc dùng công thức tổ hợp/DP.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n ≤ 7` ô chữ, mỗi ô một chữ hoa; đếm số dãy khác nhau (độ dài `1..n`) xếp từ các ô, mỗi ô dùng tối đa một lần.
2. Nắm công cụ: hoán vị (`itertools.permutations`), tập hợp để khử trùng, `Counter`, hoán vị lặp (hệ số đa thức), quy hoạch động theo chữ cái.
3. Đi từ **sinh mọi hoán vị + `set`** → **quay lui khử trùng bằng cách bỏ qua chữ giống nhau cùng tầng** → **quay lui trên bảng đếm** → **công thức hoán vị lặp** → **DP chèn chữ cái với hệ số nhị thức**.
4. Hiểu mấu chốt: mỗi **dãy phân biệt** ứng với đúng một "lần chọn chữ cái" ở mỗi vị trí; khi quay lui theo **loại chữ cái** (không theo ô) thì mỗi dãy được sinh **đúng một lần**.
5. Tránh lỗi kinh điển: đếm cả dãy trùng (hoán vị các ô), chỉ đếm dãy dùng hết ô, cộng thừa dãy rỗng, khử trùng sai kiểu (bỏ qua chữ giống nhau mà không kiểm tra ô trước đã dùng), đếm tập chữ thay vì dãy.

## 2. Đề bài & ví dụ

Bạn có `n` ô chữ `tiles`, mỗi ô in một chữ cái `tiles[i]`. Trả về **số dãy chữ cái không rỗng** (khác nhau) có thể tạo được bằng cách dùng các chữ cái trên các ô.

**Ví dụ 1:** `tiles = "AAB"` → `8` (các dãy `"A"`, `"B"`, `"AA"`, `"AB"`, `"BA"`, `"AAB"`, `"ABA"`, `"BAA"`).

**Ví dụ 2:** `tiles = "AAABBC"` → `188`.

**Ví dụ 3:** `tiles = "V"` → `1`.

**Ràng buộc:** `1 <= tiles.length <= 7`; `tiles` chỉ gồm chữ cái in hoa.

In [1]:
import itertools
import math
import random
import time
from collections import Counter
from typing import List

examples = [("AAB", 8), ("AAABBC", 188), ("V", 1)]
for t, want in examples:
    print(f"tiles = {t!r} -> {want}")

tiles = 'AAB' -> 8
tiles = 'AAABBC' -> 188
tiles = 'V' -> 1


## 3. Kiến thức nền

### 3.1 Dãy phân biệt và hoán vị lặp

Ví dụ `"AAB"`: có `3` ô nhưng hai ô `A` giống nhau. Dãy `"AB"` chỉ tính một lần dù có hai cách chọn ô `A`. Số hoán vị phân biệt của một **đa tập** gồm `k` phần tử với số lần xuất hiện `c_1, …, c_r` là hệ số đa thức `k! / (c_1! · … · c_r!)`.

In [2]:
def multinomial(counts):
    k = sum(counts)
    res = math.factorial(k)
    for c in counts:
        res //= math.factorial(c)
    return res


print(multinomial([2, 1]), multinomial([1, 1, 1]), multinomial([3, 2, 1]))
assert multinomial([2, 1]) == 3 and multinomial([1, 1, 1]) == 6 and multinomial([3, 2, 1]) == 60

3 6 60


### 3.2 Chia theo đa tập con

Một dãy phân biệt xác định một **đa tập con** của `tiles` (cho biết mỗi chữ dùng bao nhiêu lần, không vượt quá số ô có). Với mỗi đa tập con (không rỗng) có số lần `c'_1, …, c'_r` (`0 ≤ c'_i ≤ c_i`), số dãy khác nhau dùng đúng đa tập đó là `multinomial(c')`. Vậy:

`đáp án = Σ_{đa tập con ≠ ∅} multinomial(c')`.

### 3.3 Bảng đếm chữ cái

`Counter("AAB") = {A: 2, B: 1}`. Khi quay lui theo **loại chữ cái** ("tiếp theo dùng chữ nào?") thay vì theo ô, hai ô `A` không bao giờ cho hai nhánh khác nhau, nên không sinh trùng.

## 4. Ý tưởng nền tảng / trực giác

Xây dãy từng chữ một: ở mỗi bước chọn **một loại chữ cái** còn ô (số đếm `> 0`), thêm vào cuối dãy và giảm số đếm. **Mỗi lần thêm là một dãy mới** (dãy hiện tại không rỗng, dài thêm `1`), nên đếm `1` rồi đệ quy tiếp. Vì nhánh chỉ phân biệt theo loại chữ cái, hai dãy khác nhau ở một vị trí nào đó luôn ở hai nhánh khác nhau — không đếm trùng, không bỏ sót.

## 5. Brute force — sinh mọi hoán vị rồi khử trùng bằng `set`

**Ý tưởng:** với mỗi độ dài `k = 1..n`, sinh `itertools.permutations(tiles, k)` (hoán vị `k` ô có thứ tự), nối thành chuỗi, cho vào `set`; đáp án là kích thước tập. Đúng theo định nghĩa ("dãy chữ cái khác nhau"), làm **oracle**.

In [3]:
def tiles_brute(tiles):
    seen = set()
    for k in range(1, len(tiles) + 1):
        for perm in itertools.permutations(tiles, k):
            seen.add("".join(perm))
    return len(seen)


for t, want in examples:
    assert tiles_brute(t) == want
print([tiles_brute(t) for t, _ in examples])

[8, 188, 1]


In [4]:
# Code nộp LeetCode cho Brute force (đúng; n <= 7 nên đủ nhanh: tối đa 13 699 hoán vị)
class SolutionBrute:
    def numTilePossibilities(self, tiles: str) -> int:
        seen = set()
        for k in range(1, len(tiles) + 1):
            for perm in itertools.permutations(tiles, k):
                seen.add("".join(perm))
        return len(seen)

**Độ phức tạp:** `O(n! · n)` thời gian (tổng số hoán vị `Σ n!/(n-k)!`), `O(n! · n)` bộ nhớ cho `set`.

**Nút thắt:** sinh cả các hoán vị trùng rồi mới khử; có thể không sinh trùng ngay từ đầu.

## 6. Bước đột phá

Quay lui theo **loại chữ cái** với bảng đếm: mỗi dãy được sinh đúng một lần và đếm ngay `1` ở mỗi nút. Hoặc đếm bằng công thức: duyệt đa tập con và cộng hệ số đa thức.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** quay lui sắp xếp + bỏ qua chữ giống nhau cùng tầng, **Cách 3** quay lui trên bảng đếm (đáp án chính), **Cách 4** công thức hoán vị lặp trên mọi đa tập con, **Cách 5** DP chèn chữ cái với hệ số nhị thức.

### Cách 2 — Quay lui trên các ô đã sắp xếp, khử trùng cùng tầng

**Giải thích:** sắp xếp `tiles` để các chữ giống nhau nằm cạnh nhau. Quay lui chọn một ô chưa dùng cho vị trí tiếp theo; **bỏ qua** ô `i` nếu `tiles[i] == tiles[i-1]` và ô `i-1` **chưa được dùng** (ở cùng tầng, hai ô giống nhau chỉ cần thử ô đầu tiên). Mỗi nút (dãy không rỗng) đếm `1`.

In [5]:
def tiles_sorted_skip(tiles):
    s = sorted(tiles)
    n = len(s)
    used = [False] * n

    def dfs():
        total = 0
        for i in range(n):
            if used[i]:
                continue
            if i > 0 and s[i] == s[i - 1] and not used[i - 1]:
                continue                                   # cùng tầng, ô giống nhau trước đó chưa dùng: đã thử rồi
            used[i] = True
            total += 1 + dfs()                             # dãy hiện tại + mọi cách nối thêm
            used[i] = False
        return total

    return dfs()


for t, want in examples:
    assert tiles_sorted_skip(t) == want
print([tiles_sorted_skip(t) for t, _ in examples])

[8, 188, 1]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionSortedSkip:
    def numTilePossibilities(self, tiles: str) -> int:
        s = sorted(tiles)
        n = len(s)
        used = [False] * n

        def dfs():
            total = 0
            for i in range(n):
                if used[i] or (i > 0 and s[i] == s[i - 1] and not used[i - 1]):
                    continue
                used[i] = True
                total += 1 + dfs()
                used[i] = False
            return total

        return dfs()

**Độ phức tạp:** `O(S · n)` thời gian với `S` là số dãy phân biệt (`≤ 13 699`), `O(n)` bộ nhớ.

### Cách 3 — Quay lui trên bảng đếm (đáp án chính)

**Ý tưởng:** `cnt = Counter(tiles)`. `dfs()`: với mỗi loại chữ `ch` còn `cnt[ch] > 0`: `cnt[ch] -= 1`, cộng `1 + dfs()`, hoàn trả. Không cần sắp xếp hay mảng `used`.

In [7]:
def tiles_counter(tiles):
    cnt = Counter(tiles)

    def dfs():
        total = 0
        for ch in cnt:
            if cnt[ch] > 0:
                cnt[ch] -= 1
                total += 1 + dfs()                         # 1 cho dãy vừa tạo + các dãy dài hơn nối thêm
                cnt[ch] += 1
        return total

    return dfs()


for t, want in examples:
    assert tiles_counter(t) == want
print([tiles_counter(t) for t, _ in examples])

[8, 188, 1]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionCounter:
    def numTilePossibilities(self, tiles: str) -> int:
        cnt = Counter(tiles)

        def dfs():
            total = 0
            for ch in cnt:
                if cnt[ch] > 0:
                    cnt[ch] -= 1
                    total += 1 + dfs()
                    cnt[ch] += 1
            return total

        return dfs()

**Độ phức tạp:** `O(S · r)` thời gian (`r` là số loại chữ, `≤ 7`), `O(n)` bộ nhớ (độ sâu `≤ n`).

### Cách 4 — Công thức hoán vị lặp trên mọi đa tập con

**Ý tưởng:** duyệt mọi vector `(c'_1, …, c'_r)` với `0 ≤ c'_i ≤ c_i` (`Π(c_i + 1) ≤ 2^7 = 128` đa tập con); với mỗi đa tập khác rỗng cộng `multinomial(c')`. Không sinh dãy nào.

In [9]:
def tiles_formula(tiles):
    counts = list(Counter(tiles).values())
    total = 0
    for sub in itertools.product(*(range(c + 1) for c in counts)):
        if sum(sub) > 0:
            total += multinomial(sub)
    return total


for t, want in examples:
    assert tiles_formula(t) == want
print([tiles_formula(t) for t, _ in examples])

[8, 188, 1]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionFormula:
    def numTilePossibilities(self, tiles: str) -> int:
        counts = list(Counter(tiles).values())
        fact = [math.factorial(i) for i in range(len(tiles) + 1)]
        total = 0
        for sub in itertools.product(*(range(c + 1) for c in counts)):
            k = sum(sub)
            if k:
                ways = fact[k]
                for c in sub:
                    ways //= fact[c]
                total += ways
        return total

**Độ phức tạp:** `O(Π(c_i + 1) · r)` thời gian (`≤ 128` đa tập con), `O(1)` bộ nhớ phụ. Nhanh nhất và không phụ thuộc số dãy.

### Cách 5 — DP chèn chữ cái với hệ số nhị thức

**Ý tưởng:** `dp[L]` = số dãy phân biệt độ dài `L` dùng các loại chữ đã xét. Thêm loại chữ có `c` ô: chọn `k ∈ [0, c]` bản sao rồi **chèn** vào một dãy độ dài `L` đã có; số cách chọn `k` vị trí trong dãy mới độ dài `L + k` là `C(L + k, k)`; `new[L + k] += dp[L] · C(L + k, k)`. Đáp án `Σ_{L ≥ 1} dp[L]`.

In [11]:
def tiles_dp(tiles):
    dp = [1]                                              # dp[L]: số dãy độ dài L; ban đầu chỉ có dãy rỗng
    for c in Counter(tiles).values():
        new = [0] * (len(dp) + c)
        for L, ways in enumerate(dp):
            for k in range(c + 1):
                new[L + k] += ways * math.comb(L + k, k)  # chọn k vị trí cho các chữ mới trong dãy dài L + k
        dp = new
    return sum(dp[1:])


for t, want in examples:
    assert tiles_dp(t) == want
print([tiles_dp(t) for t, _ in examples])

[8, 188, 1]


In [12]:
# Code nộp LeetCode cho Cách 5
class SolutionDP:
    def numTilePossibilities(self, tiles: str) -> int:
        dp = [1]
        for c in Counter(tiles).values():
            new = [0] * (len(dp) + c)
            for L, ways in enumerate(dp):
                for k in range(c + 1):
                    new[L + k] += ways * math.comb(L + k, k)
            dp = new
        return sum(dp[1:])

**Độ phức tạp:** `O(r · n · c)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace **Cách 3** với `tiles = "AAB"`, `cnt = {A: 2, B: 1}`. Cây quay lui (mỗi nút ghi dãy đang xây và đóng góp `1`):

```
""  cnt{A:2,B:1}
├── A   (+1)  cnt{A:1,B:1}
│   ├── AA  (+1)  cnt{A:0,B:1}
│   │   └── AAB (+1)
│   └── AB  (+1)  cnt{A:1,B:0}
│       └── ABA (+1)
└── B   (+1)  cnt{A:2,B:0}
    └── BA  (+1)  cnt{A:1,B:0}
        └── BAA (+1)
```

Có `8` nút không rỗng: `A, AA, AAB, AB, ABA, B, BA, BAA` — đúng tám dãy của đề. Hàm trace in từng nút:

In [13]:
def trace_counter(tiles):
    cnt = Counter(tiles)
    seqs = []

    def dfs(prefix, depth):
        for ch in sorted(cnt):
            if cnt[ch] > 0:
                cnt[ch] -= 1
                seqs.append(prefix + ch)
                print("  " * depth + f"{prefix + ch!r} (+1)  còn {dict((k, v) for k, v in cnt.items() if v)}")
                dfs(prefix + ch, depth + 1)
                cnt[ch] += 1

    print(f"tiles = {tiles!r}")
    dfs("", 1)
    print("  tổng =", len(seqs))
    assert len(seqs) == len(set(seqs))
    return len(seqs)


assert trace_counter("AAB") == 8
assert trace_counter("AB") == 4

tiles = 'AAB'
  'A' (+1)  còn {'A': 1, 'B': 1}
    'AA' (+1)  còn {'B': 1}
      'AAB' (+1)  còn {}
    'AB' (+1)  còn {'A': 1}
      'ABA' (+1)  còn {}
  'B' (+1)  còn {'A': 2}
    'BA' (+1)  còn {'A': 1}
      'BAA' (+1)  còn {}
  tổng = 8
tiles = 'AB'
  'A' (+1)  còn {'B': 1}
    'AB' (+1)  còn {}
  'B' (+1)  còn {'A': 1}
    'BA' (+1)  còn {}
  tổng = 4


**Trace Cách 5 (DP) với `"AAB"`:** loại `A` (`c = 2`): `dp = [1, 1, 1]` (dãy rỗng, `"A"`, `"AA"`). Loại `B` (`c = 1`): `new[0] = 1`; `new[1] = dp[0]·C(1,1) + dp[1]·C(1,0) = 1 + 1 = 2`; `new[2] = dp[1]·C(2,1) + dp[2]·C(2,0) = 2 + 1 = 3`; `new[3] = dp[2]·C(3,1) = 3`. `dp = [1, 2, 3, 3]`, tổng khác rỗng `= 2 + 3 + 3 = 8`.

In [14]:
dp = [1]
for c in Counter("AAB").values():
    new = [0] * (len(dp) + c)
    for L, w in enumerate(dp):
        for k in range(c + 1):
            new[L + k] += w * math.comb(L + k, k)
    dp = new
    print("sau một loại chữ:", dp)
assert dp == [1, 2, 3, 3] and sum(dp[1:]) == 8

sau một loại chữ: [1, 1, 1]
sau một loại chữ: [1, 2, 3, 3]


## 9. Code nộp LeetCode

Chọn **Cách 3** (quay lui trên bảng đếm). Giữ nguyên `class Solution`, `numTilePossibilities(self, tiles)`:

In [15]:
class Solution:
    def numTilePossibilities(self, tiles: str) -> int:
        cnt = Counter(tiles)                        # số ô còn lại của mỗi loại chữ

        def dfs() -> int:
            total = 0
            for ch in cnt:                          # chọn LOẠI chữ tiếp theo: các ô giống nhau không sinh nhánh trùng
                if cnt[ch] > 0:
                    cnt[ch] -= 1
                    total += 1 + dfs()              # dãy vừa tạo (không rỗng) + mọi dãy dài hơn bắt đầu bằng nó
                    cnt[ch] += 1
            return total

        return dfs()


print([Solution().numTilePossibilities(t) for t, _ in examples])
assert all(Solution().numTilePossibilities(t) == v for t, v in examples)

[8, 188, 1]


## 10. Vì sao đúng

**(a) Song ánh giữa dãy và đường đi.** Mỗi dãy phân biệt `w = w_1 … w_k` (không rỗng, khả thi: mỗi chữ dùng không quá số ô có) tương ứng **đúng một** đường đi từ gốc trong cây quay lui: bước `t` chọn loại chữ `w_t`. Đường đi này hợp lệ vì `cnt[w_t] > 0` tại thời điểm đó khi và chỉ khi `w` khả thi (số lần `w_t` đã dùng chưa vượt số ô). Ngược lại mỗi nút không rỗng của cây ứng với một dãy khả thi. Hai đường đi khác nhau cho hai dãy khác nhau (khác nhau ở bước đầu tiên chọn khác loại chữ). Vậy số nút không rỗng = số dãy phân biệt khả thi.

**(b) Đếm đúng.** `dfs()` trả về số nút không rỗng trong cây con (mọi nút con ở độ sâu `≥ 1` tính từ nút hiện tại): với mỗi loại `ch` khả dụng, nút con `ch` đóng góp `1` cộng số nút trong cây con của nó `dfs()`. Đệ quy kết thúc vì tổng `cnt` giảm `1` mỗi bước.

**(c) Công thức (Cách 4).** Dãy phân biệt dùng đúng đa tập `c'` có `multinomial(c')` hoán vị phân biệt; các đa tập khác nhau cho các dãy khác nhau; tổng trên mọi đa tập con khác rỗng. Đa tập con khả thi ⇔ `c'_i ≤ c_i`.

**(d) DP (Cách 5).** Với các loại chữ đã xét, `dp[L]` là số dãy phân biệt độ dài `L`. Thêm loại chữ `x` dùng `k` bản sao: ghép dãy cũ độ dài `L` với `k` chữ `x` là chọn `k` trong `L + k` vị trí cho `x` (các chữ còn lại giữ thứ tự cũ) — `C(L + k, k)` cách, mỗi cách cho một dãy mới **khác nhau** (vị trí của `x` khác nhau hoặc dãy cũ khác nhau cho dãy mới khác nhau), và mọi dãy mới đều xuất hiện đúng một lần (bỏ các `x` đi ra dãy cũ duy nhất).

**(e) Cách 2 khử trùng.** Sau khi sắp xếp, ô `i` giống ô `i-1` mà `i-1` chưa dùng bị bỏ qua: trong mỗi tầng chỉ ô **đầu tiên** của mỗi nhóm giống nhau (chưa dùng) được thử, tương đương chọn loại chữ; khi ô `i-1` đã dùng, ô `i` là bản sao kế tiếp nên được phép. Số nhánh bằng số loại chữ khả dụng — giống Cách 3.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp brute (hoán vị + `set`) trên mọi chuỗi `{A,B,C}` độ dài `≤ 6` và mọi chuỗi `{A,B}` độ dài `7` (mục 12), và các cách khớp công thức đóng khi tất cả chữ khác nhau.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hoán vị + `set` | `O(n! · n)` | `O(n! · n)` | oracle |
| 2. Quay lui sắp xếp + bỏ trùng | `O(S · n)` | `O(n)` | kiểu "Permutations II" |
| 3. Quay lui trên bảng đếm | `O(S · r)` | `O(n)` | **đáp án chính** |
| 4. Công thức hoán vị lặp | `O(Π(c_i + 1))` | `O(1)` | nhanh nhất |
| 5. DP chèn chữ cái | `O(r · n · c)` | `O(n)` | đa thức theo `n` |

`S ≤ Σ_{k=1}^{7} 7!/(7-k)! = 13 699` (khi mọi chữ khác nhau). Đo thực tế:

In [16]:
def bench(fn, arg, repeat=5):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


for t in ("ABCDEFG", "AAABBCC", "AAAAAAA"):
    print(t)
    for name, fn in [("brute", tiles_brute), ("sorted_skip", tiles_sorted_skip), ("counter", tiles_counter), ("formula", tiles_formula), ("dp", tiles_dp)]:
        print(f"  {name:12s}: {bench(fn, t) * 1000:8.3f} ms -> {fn(t)}")

ABCDEFG
  brute       :    2.027 ms -> 13699
  sorted_skip :    3.985 ms -> 13699
  counter     :    6.986 ms -> 13699
  formula     :    0.066 ms -> 13699
  dp          :    0.009 ms -> 13699
AAABBCC
  brute       :    1.677 ms -> 649
  sorted_skip :    0.190 ms -> 649
  counter     :    0.227 ms -> 649
  formula     :    0.013 ms -> 649
  dp          :    0.005 ms -> 649
AAAAAAA
  brute       :    1.579 ms -> 7
  sorted_skip :    0.004 ms -> 7
  counter     :    0.003 ms -> 7
  formula     :    0.003 ms -> 7
  dp          :    0.002 ms -> 7


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi chuỗi nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "sorted_skip": tiles_sorted_skip,
    "counter": tiles_counter,
    "formula": tiles_formula,
    "dp": tiles_dp,
    "Solution": lambda t: Solution().numTilePossibilities(t),
    "SolutionSortedSkip": lambda t: SolutionSortedSkip().numTilePossibilities(t),
    "SolutionCounter": lambda t: SolutionCounter().numTilePossibilities(t),
    "SolutionFormula": lambda t: SolutionFormula().numTilePossibilities(t),
    "SolutionDP": lambda t: SolutionDP().numTilePossibilities(t),
}
BRUTE = {"brute": tiles_brute, "SolutionBrute": lambda t: SolutionBrute().numTilePossibilities(t)}


def check(tiles, want=None):
    if want is None:
        want = tiles_brute(tiles)
    for name, fn in {**IMPLS, **BRUTE}.items():
        got = fn(tiles)
        assert got == want, (name, tiles, got, want)
    return want


for t, want in examples:
    check(t, want)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng; 9 cách nhanh + 2 vét cạn


### B. Edge case

- Một ô: `"V"` → `1`.
- Toàn chữ giống nhau `"AAAAAAA"`: `n` dãy (`A`, `AA`, …).
- Toàn chữ khác nhau: `Σ n!/(n-k)!` (`"ABC"` → `15`, `"ABCDEFG"` → `13 699`).
- Hai chữ: `"AB"` → `4` (`A, B, AB, BA`), `"AA"` → `2`.
- Chữ giống nhau xen kẽ: `"ABA"`, `"AABB"`.
- Số lần xuất hiện cao: `"AAAAAAB"`.

In [18]:
edge = [
    ("V", 1), ("AA", 2), ("AB", 4), ("ABC", 15), ("AAA", 3), ("AAB", 8), ("ABA", 8), ("AABB", 18),
    ("AAAAAAA", 7), ("ABCDEFG", 13699), ("AAAAAAB", None), ("ABCDEFA", None), ("AAABBBC", None),
]
for t, want in edge:
    got = check(t, want)
    print(f"  {t!r:10} -> {got}")
assert Solution().numTilePossibilities("ABCDEFG") == sum(math.perm(7, k) for k in range(1, 8)) == 13699
print(len(edge), "edge case: mọi cài đặt khớp")

  'V'        -> 1
  'AA'       -> 2
  'AB'       -> 4
  'ABC'      -> 15
  'AAA'      -> 3
  'AAB'      -> 8
  'ABA'      -> 8
  'AABB'     -> 18
  'AAAAAAA'  -> 7
  'ABCDEFG'  -> 13699
  'AAAAAAB'  -> 34


  'ABCDEFA'  -> 7012


  'AAABBBC'  -> 447
13 edge case: mọi cài đặt khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_count_all_permutations(tiles):                  # đếm hoán vị các ô, không khử trùng
    return sum(1 for k in range(1, len(tiles) + 1) for _ in itertools.permutations(tiles, k))


def bug_full_length_only(tiles):                        # chỉ đếm dãy dùng hết mọi ô
    return len(set(itertools.permutations(tiles)))


def bug_include_empty(tiles):                           # cộng cả dãy rỗng
    return Solution().numTilePossibilities(tiles) + 1


def bug_skip_equal_always(tiles):                       # bỏ qua chữ giống ô trước bất kể ô trước đã dùng chưa
    s = sorted(tiles)
    n = len(s)
    used = [False] * n

    def dfs():
        total = 0
        for i in range(n):
            if used[i] or (i > 0 and s[i] == s[i - 1]):
                continue
            used[i] = True
            total += 1 + dfs()
            used[i] = False
        return total

    return dfs()


def bug_distinct_letters_only(tiles):                   # đếm tập chữ (đa tập con), không đếm thứ tự
    counts = list(Counter(tiles).values())
    return sum(1 for sub in itertools.product(*(range(c + 1) for c in counts)) if sum(sub) > 0)


def bug_multinomial_full_only(tiles):                   # chỉ công thức cho cả đa tập
    return multinomial(list(Counter(tiles).values()))


BUGS = [bug_count_all_permutations, bug_full_length_only, bug_include_empty, bug_skip_equal_always, bug_distinct_letters_only, bug_multinomial_full_only]
pool = ["".join(t) for n in range(1, 6) for t in itertools.product("ABC", repeat=n)]
for bug in BUGS:
    found = next((t for t in pool if bug(t) != tiles_brute(t)), None)
    assert found, bug.__name__
    print(f"{bug.__name__:28s} sai trên {found!r}: ra {bug(found)}, đúng {tiles_brute(found)}")

bug_count_all_permutations   sai trên 'AA': ra 4, đúng 2
bug_full_length_only         sai trên 'AA': ra 1, đúng 2
bug_include_empty            sai trên 'A': ra 2, đúng 1
bug_skip_equal_always        sai trên 'AA': ra 1, đúng 2
bug_distinct_letters_only    sai trên 'AB': ra 3, đúng 4
bug_multinomial_full_only    sai trên 'AA': ra 1, đúng 2


### D. Vét cạn mọi chuỗi nhỏ

Mọi chuỗi trên `{A, B, C}` độ dài `1..6` (`1 092` chuỗi) và mọi chuỗi trên `{A, B}` độ dài `7` (`128` chuỗi). So mọi cài đặt với brute (hoán vị + `set`).

In [20]:
t0 = time.perf_counter()
cnt = 0
for alpha, lengths in (("ABC", range(1, 7)), ("AB", [7])):
    for n in lengths:
        for tup in itertools.product(alpha, repeat=n):
            t = "".join(tup)
            want = tiles_brute(t)
            for name, fn in IMPLS.items():
                assert fn(t) == want, (name, t)
            cnt += 1
print(cnt, "chuỗi:", len(IMPLS), "cài đặt khớp brute;", round(time.perf_counter() - t0, 1), "s")

1220 chuỗi: 9 cài đặt khớp brute; 0.8 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Chuỗi ngẫu nhiên (độ dài `1..7`, bảng chữ `1..7` chữ): mọi cách khớp brute; (2) **hoán vị thứ tự ô** và **đổi tên chữ cái** không đổi đáp án; (3) **thêm một ô** làm đáp án **tăng thực sự**; (4) **chặn trên**: `ans ≤ Σ P(n, k)` (mọi chữ khác nhau), `ans ≥ n`; (5) **tập dãy sinh ra**: mọi dãy do quay lui sinh có `Counter(dãy) ⊆ Counter(tiles)`, đôi một khác nhau, số lượng = đáp án; (6) mọi chữ khác nhau ⇒ `ans = Σ P(n, k)`.

In [21]:
def all_sequences(tiles):
    cnt = Counter(tiles)
    out = []

    def dfs(prefix):
        for ch in sorted(cnt):
            if cnt[ch] > 0:
                cnt[ch] -= 1
                out.append(prefix + ch)
                dfs(prefix + ch)
                cnt[ch] += 1

    dfs("")
    return out


rng = random.Random(20240179)
t0 = time.perf_counter()
for it in range(1500):
    n = rng.randint(1, 7)
    alpha = "ABCDEFGHIJ"[:rng.randint(1, 7)]
    t = "".join(rng.choice(alpha) for _ in range(n))
    ans = check(t)                                                                  # (1)
    sh = list(t)
    rng.shuffle(sh)
    assert Solution().numTilePossibilities("".join(sh)) == ans                      # (2)
    perm = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
    rng.shuffle(perm)
    tr = str.maketrans("ABCDEFGHIJKLMNOPQRSTUVWXYZ", "".join(perm))
    assert Solution().numTilePossibilities(t.translate(tr)) == ans                  # (2)
    if n < 7:
        assert Solution().numTilePossibilities(t + rng.choice("ABCDEFGHIJ")) > ans  # (3)
    assert n <= ans <= sum(math.perm(n, k) for k in range(1, n + 1))                # (4)
    seqs = all_sequences(t)
    assert len(seqs) == len(set(seqs)) == ans                                       # (5)
    c0 = Counter(t)
    assert all(not (Counter(s) - c0) and s for s in seqs)
    if len(set(t)) == n:
        assert ans == sum(math.perm(n, k) for k in range(1, n + 1))                 # (6)
print("1500 chuỗi ngẫu nhiên: 6 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

1500 chuỗi ngẫu nhiên: 6 tính chất đúng; 3.8 s


### F. Giới hạn

`n = 7`: mọi chữ khác nhau (`13 699`), toàn một chữ (`7`), hai loại (`AAAAAAB`, `AAABBBB`), ba loại `AAABBCC`, bảy loại có lặp. Các cách phải chạy nhanh.

In [22]:
t0 = time.perf_counter()
for t in ["ABCDEFG", "AAAAAAA", "AAAAAAB", "AAABBBB", "AAABBCC", "AABBCCD", "ABCDEFA", "XYZXYZX", "ZZZZZZY"]:
    ans = check(t)
    print(f"  {t}: {ans}")
print("n = 7:", round(time.perf_counter() - t0, 2), "s")
assert Solution().numTilePossibilities("AAAAAAA") == 7

  ABCDEFG: 13699
  AAAAAAA: 7
  AAAAAAB: 34
  AAABBBB: 124
  AAABBCC: 649
  AABBCCD: 1840
  ABCDEFA: 7012
  XYZXYZX: 649
  ZZZZZZY: 34
n = 7: 0.14 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm hoán vị các ô thay vì dãy phân biệt

In [23]:
print("bug:", bug_count_all_permutations("AAB"), "| đúng:", Solution().numTilePossibilities("AAB"))
assert bug_count_all_permutations("AAB") == 15 and Solution().numTilePossibilities("AAB") == 8

bug: 15 | đúng: 8


**Vì sao sai:** hai ô `A` là **hai ô khác nhau** nhưng cho cùng một dãy. Phải khử trùng (bằng `set`, bằng quay lui theo loại chữ, hoặc bằng công thức).

### Lỗi 2: chỉ đếm dãy dùng hết ô

In [24]:
print("bug:", bug_full_length_only("AAB"), "| đúng:", Solution().numTilePossibilities("AAB"))
assert bug_full_length_only("AAB") == 3

bug: 3 | đúng: 8


**Vì sao sai:** đề cho phép dùng **một phần** các ô: dãy độ dài `1..n`. `"AAB"` có `3` dãy dùng hết ô nhưng `8` dãy tổng cộng.

### Lỗi 3: cộng cả dãy rỗng

In [25]:
print("bug:", bug_include_empty("V"), "| đúng:", Solution().numTilePossibilities("V"))
assert bug_include_empty("V") == 2 and Solution().numTilePossibilities("V") == 1

bug: 2 | đúng: 1


**Vì sao sai:** đề hỏi dãy **không rỗng**. Nếu đếm `1` cho nút gốc (dãy rỗng) hoặc cộng `Σ` cả `L = 0` thì lệch `1`.

### Lỗi 4: khử trùng sai kiểu (bỏ qua chữ giống ô trước bất kể đã dùng)

In [26]:
print("bug:", bug_skip_equal_always("AAB"), "| đúng:", Solution().numTilePossibilities("AAB"))
assert bug_skip_equal_always("AAB") < Solution().numTilePossibilities("AAB")

bug: 4 | đúng: 8


**Vì sao sai:** bỏ qua `s[i] == s[i-1]` **mà không xét `used[i-1]`** làm mất mọi dãy chứa hai chữ giống nhau (`"AA"`, `"AAB"`...). Điều kiện đúng: bỏ qua **chỉ khi ô `i-1` chưa được dùng** (cùng tầng).

### Lỗi 5: đếm tập chữ thay vì dãy

In [27]:
print("bug:", bug_distinct_letters_only("AAB"), "| đúng:", Solution().numTilePossibilities("AAB"))
assert bug_distinct_letters_only("AAB") == 5 and Solution().numTilePossibilities("AAB") == 8

bug: 5 | đúng: 8


**Vì sao sai:** đa tập con `{A, B}` cho hai dãy `"AB"` và `"BA"`; phải nhân với số hoán vị lặp của đa tập đó.

### Lỗi 6 (bonus): chỉ công thức cho cả đa tập

In [28]:
print("bug:", bug_multinomial_full_only("AAB"), "| đúng:", Solution().numTilePossibilities("AAB"))
assert bug_multinomial_full_only("AAB") == 3

bug: 3 | đúng: 8


**Vì sao sai:** `multinomial` của cả đa tập chỉ đếm dãy dùng **mọi** ô; phải cộng thêm các đa tập con thực sự nhỏ hơn.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đếm các dãy/hoán vị **phân biệt** từ một đa tập (có phần tử trùng)" → **quay lui theo loại phần tử với bảng đếm** hoặc công thức hoán vị lặp.

**Mẹo:**

1. Quay lui theo **loại** (khoá của `Counter`), không theo ô; khi đó không sinh trùng.
2. Mỗi nút không rỗng của cây là một dãy hợp lệ: cộng `1` rồi đệ quy.
3. Cách khác: sắp xếp và `if i > 0 and s[i] == s[i-1] and not used[i-1]: continue`.
4. Đếm bằng công thức: `Σ multinomial(c')` trên mọi đa tập con khác rỗng.
5. DP chèn chữ cái: `new[L+k] += dp[L] · C(L+k, k)`.
6. Oracle: `set` của mọi hoán vị (`n ≤ 7`).

**Họ bài cùng pattern:** Permutations II (47), Subsets II (90), Combination Sum II (40), Letter Combinations of a Phone Number (17), Generate Parentheses (22), Number of Squareful Arrays (996).

## 15. Biến thể

1. **Số dãy dài đúng `k`:** hệ số `dp[k]` của DP (hoặc quay lui giới hạn độ sâu).
2. **Chỉ dùng hết ô:** `multinomial` toàn bộ (hoán vị phân biệt, bài Permutations II).
3. **Không hai chữ giống nhau liền kề:** quay lui có nhớ chữ cuối cùng.
4. **Liệt kê** tất cả dãy theo thứ tự từ điển.

Cài đặt và đối chiếu brute:

In [29]:
def count_length_k(tiles):
    dp = [1]
    for c in Counter(tiles).values():
        new = [0] * (len(dp) + c)
        for L, ways in enumerate(dp):
            for k in range(c + 1):
                new[L + k] += ways * math.comb(L + k, k)
        dp = new
    return dp                                             # dp[k] = số dãy độ dài k


def no_adjacent_equal(tiles):
    cnt = Counter(tiles)

    def dfs(last):
        total = 0
        for ch in cnt:
            if cnt[ch] > 0 and ch != last:
                cnt[ch] -= 1
                total += 1 + dfs(ch)
                cnt[ch] += 1
        return total

    return dfs("")


def brute_variants(tiles):
    seqs = {"".join(p) for k in range(1, len(tiles) + 1) for p in itertools.permutations(tiles, k)}
    by_len = Counter(len(s) for s in seqs)
    no_adj = sum(1 for s in seqs if all(s[i] != s[i + 1] for i in range(len(s) - 1)))
    full = sum(1 for s in seqs if len(s) == len(tiles))
    return by_len, no_adj, full, sorted(seqs)


rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 7)
    t = "".join(rng.choice("ABCD"[:rng.randint(1, 4)]) for _ in range(n))
    by_len, no_adj, full, seqs = brute_variants(t)
    dp = count_length_k(t)
    assert all(dp[k] == by_len[k] for k in range(1, n + 1))
    assert no_adjacent_equal(t) == no_adj
    assert multinomial(list(Counter(t).values())) == full
    assert all_sequences(t) != [] and sorted(all_sequences(t)) == seqs
print("số dãy theo độ dài / không liền kề giống nhau / dùng hết ô / liệt kê: khớp brute")

số dãy theo độ dài / không liền kề giống nhau / dùng hết ô / liệt kê: khớp brute


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị + `set` | khử trùng sau | `O(n! n)` | `O(n! n)` | oracle |
| 2. Quay lui sắp xếp | bỏ qua bản sao cùng tầng | `O(S n)` | `O(n)` | quen thuộc từ Permutations II |
| 3. Quay lui bảng đếm | chọn loại chữ | `O(S r)` | `O(n)` | **đáp án chính** |
| 4. Công thức | `Σ multinomial` | `O(Π(c_i+1))` | `O(1)` | nhanh nhất |
| 5. DP chèn | `C(L+k, k)` | `O(r n c)` | `O(n)` | mở rộng cho `n` lớn |

Mọi cách khớp brute trên `1 220` chuỗi nhỏ (`{A,B,C}` `≤ 6`, `{A,B}` `= 7`) và `1 500` chuỗi ngẫu nhiên.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().numTilePossibilities("AB"), Solution().numTilePossibilities("AAA"), Solution().numTilePossibilities("ABC"), Solution().numTilePossibilities("AABB"))

4 3 15 18


### Bài 2 (tính tay)

`tiles = "AABB"`. Liệt kê số dãy theo độ dài `1, 2, 3, 4` (dùng `multinomial` cho từng đa tập con) rồi cộng lại. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện quay lui trên bảng đếm:

In [31]:
def tiles_todo(tiles):
    cnt = Counter(tiles)

    def dfs():
        total = 0
        for ch in cnt:
            pass        # TODO 1: nếu cnt[ch] > 0: giảm cnt[ch], total += 1 + dfs(), hoàn trả cnt[ch]
        return total

    return dfs()


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(tiles_todo(t) == v for t, v in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **công thức hoán vị lặp** trên mọi đa tập con:

In [32]:
class MySolution:
    def numTilePossibilities(self, tiles):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().numTilePossibilities(t) == v for t, v in examples)

### Bài 5 (mini challenge): đếm dãy dài đúng `k` không có hai chữ giống nhau liền kề

Viết `count_no_adjacent(tiles, k)` trả về số dãy phân biệt **dài đúng `k`** từ các ô mà không có hai chữ cái giống nhau đứng cạnh nhau. Ví dụ `count_no_adjacent("AAB", 3) == 1` (`"ABA"`), `count_no_adjacent("AAB", 2) == 2` (`"AB"`, `"BA"`).

In [33]:
def count_no_adjacent(tiles, k):
    pass            # TODO


# Khi làm xong:
# assert count_no_adjacent("AAB", 3) == 1 and count_no_adjacent("AAB", 2) == 2 and count_no_adjacent("AAB", 1) == 2

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `"AB"`: `A, B, AB, BA`.
- Mức 2: `"AAA"`: `A, AA, AAA`; `"ABC"`: `3 + 6 + 6`.
- Mức 3: `4 3 15 18`. Với `"AABB"`: độ dài 1: `2`; độ dài 2: `AA, BB, AB, BA = 4`; độ dài 3: `AAB` có `3`, `ABB` có `3` ⇒ `6`; độ dài 4: `6`; tổng `18`.

**Bài 2**

- Mức 1: đa tập con `(a, b)` với `0 ≤ a, b ≤ 2`, `(a, b) ≠ (0, 0)`.
- Mức 2: `multinomial(a, b) = (a + b)! / (a! b!)`.
- Mức 3: độ dài 1: `(1,0)=1`, `(0,1)=1` ⇒ `2`; độ dài 2: `(2,0)=1`, `(1,1)=2`, `(0,2)=1` ⇒ `4`; độ dài 3: `(2,1)=3`, `(1,2)=3` ⇒ `6`; độ dài 4: `(2,2)=6`; tổng `18`.

**Bài 3**

- Mức 1: `for ch in cnt: if cnt[ch] > 0:`.
- Mức 2: `cnt[ch] -= 1`; `total += 1 + dfs()`.
- Mức 3: `cnt[ch] += 1` để hoàn trả.

**Bài 4**

- Mức 1: `counts = list(Counter(tiles).values())`.
- Mức 2: `itertools.product(*(range(c + 1) for c in counts))`.
- Mức 3: với mỗi `sub` có `sum(sub) > 0`, cộng `factorial(sum(sub)) // Π factorial(c')`.

**Bài 5**

- Mức 1: quay lui trên bảng đếm có tham số `last` (chữ cuối) và `length`.
- Mức 2: chỉ chọn `ch != last`; khi `length == k` cộng `1` và dừng.
- Mức 3: `dfs("", 0)`; kiểm: `"AAB"`, `k = 3` chỉ có `"ABA"`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().numTilePossibilities("AB"), Solution().numTilePossibilities("AAA"), Solution().numTilePossibilities("ABC"), Solution().numTilePossibilities("AABB"))
print(vals)
assert vals == (4, 3, 15, 18)
assert vals == (tiles_brute("AB"), tiles_brute("AAA"), tiles_brute("ABC"), tiles_brute("AABB"))
print("Bài 1 OK")

(4, 3, 15, 18)
Bài 1 OK


### Bài 2

In [35]:
by_len = [0] * 5
for a in range(3):
    for b in range(3):
        if a + b:
            by_len[a + b] += multinomial([a, b])
print(by_len)
assert by_len == [0, 2, 4, 6, 6] and sum(by_len) == 18 == tiles_brute("AABB")
assert count_length_k("AABB") == [1, 2, 4, 6, 6]
print("Bài 2 OK")

[0, 2, 4, 6, 6]
Bài 2 OK


### Bài 3

In [36]:
def tiles_answer(tiles):
    cnt = Counter(tiles)

    def dfs():
        total = 0
        for ch in cnt:
            if cnt[ch] > 0:
                cnt[ch] -= 1
                total += 1 + dfs()
                cnt[ch] += 1
        return total

    return dfs()


assert all(tiles_answer(t) == v for t, v in examples)
for t in pool:
    assert tiles_answer(t) == tiles_brute(t)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def numTilePossibilities(self, tiles):
        counts = list(Counter(tiles).values())
        total = 0
        for sub in itertools.product(*(range(c + 1) for c in counts)):
            k = sum(sub)
            if k:
                ways = math.factorial(k)
                for c in sub:
                    ways //= math.factorial(c)
                total += ways
        return total


assert all(MySolutionAnswer().numTilePossibilities(t) == v for t, v in examples)
for t in pool:
    assert MySolutionAnswer().numTilePossibilities(t) == tiles_brute(t)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def count_no_adjacent(tiles, k):
    cnt = Counter(tiles)

    def dfs(last, length):
        if length == k:
            return 1
        total = 0
        for ch in cnt:
            if cnt[ch] > 0 and ch != last:
                cnt[ch] -= 1
                total += dfs(ch, length + 1)
                cnt[ch] += 1
        return total

    return dfs("", 0)


assert count_no_adjacent("AAB", 3) == 1 and count_no_adjacent("AAB", 2) == 2 and count_no_adjacent("AAB", 1) == 2
for t in pool[:300]:
    seqs = {"".join(p) for kk in range(1, len(t) + 1) for p in itertools.permutations(t, kk)}
    for k in range(1, len(t) + 1):
        want = sum(1 for s in seqs if len(s) == k and all(s[i] != s[i + 1] for i in range(k - 1)))
        assert count_no_adjacent(t, k) == want, (t, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm dãy phân biệt không rỗng từ đa tập, mỗi phần tử dùng ≤ số lần có):**

```python
class Solution:
    def numTilePossibilities(self, tiles: str) -> int:
        cnt = Counter(tiles)

        def dfs() -> int:
            total = 0
            for ch in cnt:
                if cnt[ch] > 0:
                    cnt[ch] -= 1
                    total += 1 + dfs()
                    cnt[ch] += 1
            return total

        return dfs()
```

**Checklist trước khi submit:**

- [ ] Quay lui theo **loại chữ** (khoá của `Counter`), không theo ô, để khỏi sinh trùng?
- [ ] Mỗi nút không rỗng cộng `1` (không cộng dãy rỗng)?
- [ ] Hoàn trả `cnt[ch] += 1` sau khi đệ quy?
- [ ] Nếu dùng sắp xếp: điều kiện bỏ qua có `not used[i-1]`?
- [ ] Đã thử `"AAB" → 8`, `"AAABBC" → 188`, `"V" → 1`, toàn chữ khác nhau?
- [ ] Đúng tên `class Solution`, `numTilePossibilities(self, tiles)`, không in debug.